<a href="https://colab.research.google.com/github/PARIJAAT-13/Flyrank-A.I/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PARIJAAT-13/Flyrank-A.I/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Data contract

- **One row means:** one pseudonymized content item/page.
- **Time window:** the row contains activity aggregated over a trailing 90-day window ending at export time, with separate recent 30-day and previous 30-day comparison windows.
- **What I want to predict/rank:** whether a content item is declining in impressions, using `trend_direction == "down"` as the label definition.
- **Main data source:** the available content-refresh dataset, with content, keyword, and search-performance fields.
- **Deliberately excluded:** `trend_direction` and `trend_pct` are excluded from model features because they directly define or describe the outcome and would cause leakage.

In [ ]:
import pandas as pd
from pathlib import Path

repo_root = Path("/content/Flyrank-A.I")
data_path = repo_root / "data/raw/content_refresh_anonymized.csv"

if not data_path.exists():
    !git clone https://github.com/PARIJAAT-13/Flyrank-A.I.git /content/Flyrank-A.I
    repo_root = Path("/content/Flyrank-A.I")
    data_path = repo_root / "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("One row represents: one content item/page")
print("Time basis: trailing 90-day window ending at export time")
print("Label source: trend_direction == 'down'")

Rows: 30000
Columns: 44
One row represents: one content item/page
Time basis: trailing 90-day window ending at export time
Label source: trend_direction == 'down'


In [ ]:
print("=== AVAILABLE DUCKDB TABLES ===")

try:
    display(con.execute("SHOW TABLES").df())
except Exception as e:
    print("SHOW TABLES failed:", e)

print("\n=== AVAILABLE VIEWS ===")

try:
    display(con.execute("SHOW VIEWS").df())
except Exception as e:
    print("SHOW VIEWS failed:", e)

=== AVAILABLE DUCKDB TABLES ===


,name



=== AVAILABLE VIEWS ===
SHOW VIEWS failed: Catalog Error: Table with name VIEWS does not exist!

LINE 1: SHOW VIEWS
             ^


In [ ]:
print("\n=== NOTEBOOK VARIABLES ===")

for name in sorted([
    n for n in globals()
    if not n.startswith("_")
]):
    obj = globals()[name]
    print(f"{name}: {type(obj).__name__}")


=== NOTEBOOK VARIABLES ===
In: list
Out: dict
Path: type
availability_df: DataFrame
availability_fields: list
availability_results: list
available: int64
col: str
con: DuckDBPyConnection
context_fields: list
data_path: PosixPath
dataframes: dict
df: DataFrame
df_limit: DataFrame
duckdb: module
duplicate_content_ids: int64
excluded_fields: list
exit: ZMQExitAutocall
feature_fields: list
get_ipython: method
label_field: str
missing: Series
missing_contract_fields: list
name: str
obj: str
pd: module
preferred_names: list
quit: ZMQExitAutocall
repo_root: PosixPath
unavailable: int64
window_fields: list


### Fields: feature / label / context / excluded

**Features:**  
- `search_volume` — search-demand signal available in the dataset.
- `competition` — keyword competition signal.
- `cpc` — keyword commercial-value signal.
- `impressions_90d` — historical search visibility.
- `clicks_90d` — historical search clicks.
- `pageviews_90d` — historical page views.
- `sessions_90d` — historical sessions.
- `engaged_sessions_90d` — historical engaged sessions.
- `impressions_last_30d` — recent search visibility.
- `clicks_last_30d` — recent search clicks.
- `sessions_last_30d` — recent sessions.
- `impressions_prev_30d` — previous-period search visibility.
- `clicks_prev_30d` — previous-period search clicks.
- `sessions_prev_30d` — previous-period sessions.
- `content_age_days` — content age at the observed snapshot.
- `days_since_last_update` — time since the content was last updated.
- `ctr` — observed click-through rate.
- `avg_position` — observed average search position.

**Label:**  
- `trend_direction` — the target used to define whether impressions are declining; `"down"` is the decline class.

**Context:**  
- `content_id` — pseudonymized content-item identifier.
- `client_id` — pseudonymized client grouping identifier.
- `content_type` — content category.
- `main_intent` — search/content intent.
- `age_tier`, `age_tier_order` — content-age grouping.
- `freshness_tier` — freshness grouping.
- `word_count_tier`, `char_count_tier` — content-size groupings.
- `impression_tier`, `position_tier` — descriptive performance groupings.
- `trend_pct` — descriptive outcome magnitude, retained as context but not used as a feature.

**Excluded:**  
- `trend_direction` — excluded from model features because it directly defines the target.
- `trend_pct` — excluded from model features because it describes the outcome.
- `provider_used` and `model_used` — excluded because they are not required for the core search-performance contract and contain substantial missingness.
- `scroll_rate` and `ai_traffic_pct` — excluded from the core feature set because they are downstream engagement/traffic signals rather than necessary search-demand inputs.

The feature set is intentionally limited to signals that can be treated as available at the decision point. Outcome-defining fields are not used as model inputs.

In [ ]:
# ML-04 — Section 2: field contract

feature_fields = [
    "search_volume",
    "competition",
    "cpc",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "engaged_sessions_90d",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
]

label_field = "trend_direction"

context_fields = [
    "content_id",
    "client_id",
    "content_type",
    "main_intent",
    "age_tier",
    "age_tier_order",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier",
    "trend_pct",
]

excluded_fields = [
    "trend_direction",
    "trend_pct",
    "provider_used",
    "model_used",
    "scroll_rate",
    "ai_traffic_pct",
]

print("=== FIELD CONTRACT ===")
print("Features:", len(feature_fields))
print("Label:", label_field)
print("Context:", len(context_fields))
print("Excluded:", len(excluded_fields))

missing_contract_fields = [
    c for c in feature_fields + [label_field] + context_fields + excluded_fields
    if c not in df.columns
]

print("\nMissing contract fields:", missing_contract_fields)

assert not missing_contract_fields, "A contract field is missing from the dataset."

print("✓ All contract fields exist in the dataset.")

=== FIELD CONTRACT ===
Features: 18
Label: trend_direction
Context: 12
Excluded: 6

Missing contract fields: []
✓ All contract fields exist in the dataset.


### 3. Verify the data contract

I verify the contract using three checks on the available content-refresh snapshot:

1. **Grain:** confirm that each row represents a pseudonymized content item/page and check the content identifier for missing or duplicate values.
2. **Window structure:** verify that the dataset contains the stated trailing 90-day activity window and separate recent and previous 30-day comparison windows.
3. **Availability:** measure non-missing values for the required signals.

The provided CSV is a snapshot rather than a dated daily table, so a March 2026 row count and date span cannot be verified from this file without inventing information. The dataset also does not expose boolean availability fields, so availability is measured from observed non-null values rather than an unavailable `IS TRUE` field.

These checks are descriptive validation of the available dataset. They do not establish causality or guarantee that the same coverage or measurement quality holds for other datasets or time periods.

In [ ]:
# ML-04 — Section 3: verify the data contract
# Three required checks:
# 1. Grain
# 2. Row count + date span
# 3. Availability using IS TRUE

print("=== CHECK 1: GRAIN ===")

# The CSV is a snapshot rather than a daily event table.
# Verify that content_id is the expected page-level identifier.
print("Rows:", len(df))
print("Unique content_id:", df["content_id"].nunique())

duplicate_content_ids = df["content_id"].duplicated().sum()

print("Duplicate content_id rows:", duplicate_content_ids)

assert df["content_id"].notna().all(), "content_id contains missing values."

print("✓ content_id is present for every row.")
print("✓ Grain is one pseudonymized content item/page per row in this snapshot.")


print("\n=== CHECK 2: SLICE SIZE / DATE WINDOW ===")

# This dataset does not expose a daily date column.
# Therefore, do not invent a March 2026 date range.
# Verify the available snapshot/window fields instead.

print("Dataset rows:", len(df))
print("Dataset columns:", len(df.columns))

window_fields = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
]

print("\nWindow fields present:")
for col in window_fields:
    print(f"✓ {col}")

assert all(col in df.columns for col in window_fields)

print(
    "\n✓ The dataset exposes a trailing 90-day window "
    "plus recent and previous 30-day comparison windows."
)


print("\n=== CHECK 3: AVAILABILITY ===")

# The CSV does not contain boolean availability columns.
# We therefore construct explicit availability indicators from
# non-missing values rather than pretending an IS TRUE field exists.

availability_fields = [
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "impressions_last_30d",
    "clicks_last_30d",
    "avg_position",
]

availability_results = []

for col in availability_fields:
    available = df[col].notna().sum()
    unavailable = df[col].isna().sum()

    availability_results.append({
        "field": col,
        "available_rows": int(available),
        "unavailable_rows": int(unavailable),
        "availability_pct": round(100 * available / len(df), 2)
    })

availability_df = pd.DataFrame(availability_results)

display(availability_df)

print(
    "✓ Availability is measured explicitly from non-missing values. "
    "Unavailable measurements are not treated as zero."
)

=== CHECK 1: GRAIN ===
Rows: 30000
Unique content_id: 30000
Duplicate content_id rows: 0
✓ content_id is present for every row.
✓ Grain is one pseudonymized content item/page per row in this snapshot.

=== CHECK 2: SLICE SIZE / DATE WINDOW ===
Dataset rows: 30000
Dataset columns: 44

Window fields present:
✓ impressions_90d
✓ clicks_90d
✓ sessions_90d
✓ impressions_last_30d
✓ clicks_last_30d
✓ sessions_last_30d
✓ impressions_prev_30d
✓ clicks_prev_30d
✓ sessions_prev_30d

✓ The dataset exposes a trailing 90-day window plus recent and previous 30-day comparison windows.

=== CHECK 3: AVAILABILITY ===


,field,available_rows,unavailable_rows,availability_pct
0,search_volume,27532,2468,91.77
1,impressions_90d,30000,0,100.00
2,clicks_90d,30000,0,100.00
3,impressions_last_30d,30000,0,100.00
4,clicks_last_30d,30000,0,100.00
5,avg_position,30000,0,100.00


✓ Availability is measured explicitly from non-missing values. Unavailable measurements are not treated as zero.


## 4. Data limits

This data supports descriptive analysis of the selected Search Intelligence slice, but it has important limits.

- The available history is not necessarily balanced across all pages, clients, and time periods, so comparisons may be affected by uneven historical coverage.
- Early observations may rely more heavily on Google Search Console data, so the available fields and measurement quality may differ across the history.
- The feature and outcome windows can overlap. Therefore, a measured relationship should not automatically be interpreted as causal evidence.
- The available snapshot is used for contract verification and feature construction. Because this CSV does not expose a daily date column, the notebook does not claim a specific March 2026 row count or date span. Later data should be treated separately when evaluating future outcomes.
- Missing or unavailable measurements mean that an unavailable signal should not be interpreted as a zero value.
- Client, page, and time differences can affect observed patterns, so results from this slice should be treated as directional evidence rather than a production guarantee.

### Named limitation

The main limitation is **uneven historical coverage across entities and time windows**, which can limit how confidently results from one slice generalize to the complete warehouse.

In [ ]:
# ML-04 — Section 4: Data limits
# Safe diagnostic version: uses the dataframe already loaded in the notebook.

print("=== DATA LIMITS CHECK ===")

# Find dataframe objects already created in the notebook
dataframes = {
    name: obj
    for name, obj in globals().items()
    if not name.startswith("_") and isinstance(obj, pd.DataFrame)
}

print("DataFrames currently available:")
for name, obj in dataframes.items():
    print(f"  {name}: {obj.shape}")

if not dataframes:
    raise RuntimeError(
        "No pandas DataFrame is currently available. "
        "Run the earlier dataset-loading cells first."
    )

# Prefer the main dataset if one is clearly available
preferred_names = [
    "df",
    "data",
    "dataset",
    "warehouse_df",
]

df_limit = None

for name in preferred_names:
    if name in dataframes:
        df_limit = dataframes[name]
        print(f"\nUsing DataFrame: {name}")
        break

if df_limit is None:
    name = list(dataframes.keys())[0]
    df_limit = dataframes[name]
    print(f"\nUsing available DataFrame: {name}")

print(f"Rows: {len(df_limit):,}")
print(f"Columns: {len(df_limit.columns)}")

print("\n=== AVAILABLE COLUMNS ===")
print(list(df_limit.columns))

print("\n=== MISSING VALUES ===")
missing = (
    df_limit.isna()
    .sum()
    .sort_values(ascending=False)
)

display(missing[missing > 0].to_frame("missing_rows"))

print("\n=== DATA LIMITS ===")
print(
    "This dataset may have uneven historical coverage across "
    "entities and time periods. Missing measurements should not "
    "automatically be interpreted as zero. Overlapping feature "
    "and outcome windows can also limit causal interpretation."
)

print("\n✓ Data-limits check completed without assuming a DuckDB table name.")

=== DATA LIMITS CHECK ===
DataFrames currently available:
  df: (30000, 44)
  df_limit: (30000, 44)
  availability_df: (6, 4)

Using DataFrame: df
Rows: 30,000
Columns: 44

=== AVAILABLE COLUMNS ===
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

=== MISSING VALUES ===


,missing_rows
provider_used,21438
word_count,7699
char_count,7699
word_count_tier,7699
char_count_tier,7699
model_used,5733
trend_pct,3388
competition_level,2610
search_volume,2468
cpc,2468



=== DATA LIMITS ===
This dataset may have uneven historical coverage across entities and time periods. Missing measurements should not automatically be interpreted as zero. Overlapping feature and outcome windows can also limit causal interpretation.

✓ Data-limits check completed without assuming a DuckDB table name.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.